# MLOps Pipeline: Training, Registry & Production Inference

This notebook demonstrates a complete MLOps workflow:

## Phase 1: Experimentation & Development
- **Dataset lineage tracking** with `mlflow.log_input()` for reproducibility
- Hyperparameter tuning with MLflow experiment tracking
- Feature importance and lineage tracking
- Model comparison and selection

## Phase 2: Model Publishing
- Register best model to MLflow Model Registry
- Version management and stage transitions
- Model metadata and signatures

## Phase 3: Production Inference
- Load models from registry (not pkl files)
- Track inference performance and drift
- Compare production vs training metrics

**MLflow UI**: http://localhost:5001

## 1. Setup and Data Preparation

In [ ]:
# Install required packages
!pip install -q mlflow xgboost scikit-learn psycopg2-binary clickhouse-connect shap boto3

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from typing import Dict, List, Tuple

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Configure MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
EXPERIMENT_NAME = "fraud-detection-experiments"
MODEL_NAME = "fraud-detection-model"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

# Initialize MLflow client for registry operations
client = MlflowClient()

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Model Registry Name: {MODEL_NAME}")
print(f"\nView experiments at: http://localhost:5001")

In [ ]:
# Load real data from ClickHouse (or use synthetic if unavailable)
import clickhouse_connect

CLICKHOUSE_HOST = os.getenv('CLICKHOUSE_HOST', 'clickhouse').replace('http://', '').split(':')[0]

# Feature columns used for training - THIS IS OUR FEATURE LINEAGE
# 11 features total (8 original + 2 audit features: region, vendor + 1 time feature)
FEATURE_COLUMNS = [
    'amount',                    # 0: Transaction amount
    'amount_zscore_customer',    # 1: Z-score vs customer average
    'amount_zscore_category',    # 2: Z-score vs category average
    'customer_txn_count_log',    # 3: Log of customer transaction history
    'tx_hour',                   # 4: Hour of transaction (0-23)
    'tx_day_of_week',            # 5: Day of week (1-7)
    'type_encoded',              # 6: Transaction type (encoded)
    'channel_encoded',           # 7: Channel (encoded)
    'category_encoded',          # 8: Category (encoded)
    'region_encoded',            # 9: Region (encoded) - AUDIT FEATURE
    'vendor_encoded'             # 10: Vendor (encoded) - AUDIT FEATURE
]

FEATURE_DESCRIPTIONS = {
    'amount': 'Raw transaction amount in dollars',
    'amount_zscore_customer': 'How unusual is this amount for this customer (-5 to 5)',
    'amount_zscore_category': 'How unusual is this amount for this category (-5 to 5)',
    'customer_txn_count_log': 'Log-scaled count of customer transaction history',
    'tx_hour': 'Hour of day when transaction occurred (0-23)',
    'tx_day_of_week': 'Day of week (1=Monday, 7=Sunday)',
    'type_encoded': 'Transaction type: 0=DEBIT, 1=CREDIT, 2=TRANSFER, 3=PAYMENT',
    'channel_encoded': 'Channel: 0=mobile, 1=web, 2=atm, 3=branch',
    'category_encoded': 'Category: 0=grocery, 1=travel, 2=crypto, 3=gambling, etc.',
    'region_encoded': 'Region: 0=APAC, 1=EMEA, 2=AMER (for audit - different fraud thresholds)',
    'vendor_encoded': 'Vendor/processor: 0=Visa, 1=Mastercard, 2=Amex, etc. (for audit)'
}

# Encoding maps (shared with inference services)
TYPE_MAP = {'DEBIT': 0, 'CREDIT': 1, 'TRANSFER': 2, 'PAYMENT': 3}
CHANNEL_MAP = {'mobile': 0, 'web': 1, 'atm': 2, 'branch': 3, 'online': 1, 'Mobile App': 0, 'Online Banking': 1, 'ATM': 2, 'Branch Teller': 3}
CATEGORY_MAP = {
    'grocery': 0, 'travel': 1, 'crypto': 2, 'gambling': 3, 'shopping': 4,
    'dining': 5, 'entertainment': 6, 'utilities': 7,
    'Groceries': 0, 'Travel': 1, 'Crypto': 2, 'Gambling': 3, 'Shopping': 4,
    'Dining': 5, 'Entertainment': 6, 'Utilities': 7,
    'Salary': 8, 'Bonus': 9, 'Refund': 10, 'Transfer': 11,
    'Pharmacy': 12, 'Healthcare': 13, 'Gas': 14, 'Subscription': 15
}
REGION_MAP = {'APAC': 0, 'EMEA': 1, 'AMER': 2, 'LATAM': 3}
VENDOR_MAP = {'Visa': 0, 'Mastercard': 1, 'Amex': 2, 'Discover': 3, 'PayPal': 4, None: 0}

try:
    ch_client = clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST, port=8123,
        username='default', password='admin123'
    )
    
    # Check if we have data
    count = ch_client.query('SELECT count() FROM transactions').result_rows[0][0]
    print(f"ClickHouse transactions: {count:,}")
    
    if count > 1000:
        # Load real data with feature engineering
        # Note: is_fraud is in fraud_labels table, not transactions
        # Include region and vendor for audit features
        query = """
        SELECT
            t.tx_id,
            t.amount,
            toHour(t.created_at) as tx_hour,
            toDayOfWeek(t.created_at) as tx_day_of_week,
            COALESCE(f.is_fraud, 0) as is_fraud,
            t.type,
            t.channel,
            t.category,
            t.customer_id,
            t.merchant,
            t.region,
            t.vendor
        FROM transactions t
        LEFT JOIN fraud_labels f ON t.tx_id = f.tx_id
        ORDER BY t.tx_id DESC
        LIMIT 50000
        """
        result = ch_client.query(query)
        df = pd.DataFrame(result.result_rows, columns=result.column_names)
        
        # Feature engineering
        # Customer stats
        cust_stats = df.groupby('customer_id').agg({
            'amount': ['mean', 'std', 'count'],
            'is_fraud': 'sum'
        }).reset_index()
        cust_stats.columns = ['customer_id', 'cust_avg', 'cust_std', 'cust_count', 'cust_fraud']
        df = df.merge(cust_stats, on='customer_id', how='left')
        
        # Category stats
        cat_stats = df.groupby('category').agg({
            'amount': ['mean', 'std'],
            'is_fraud': ['sum', 'mean']
        }).reset_index()
        cat_stats.columns = ['category', 'cat_avg', 'cat_std', 'cat_fraud', 'category_fraud_rate']
        cat_stats['category_fraud_rate'] = cat_stats['category_fraud_rate'] * 100
        df = df.merge(cat_stats, on='category', how='left')
        
        # Compute z-scores
        df['amount_zscore_customer'] = ((df['amount'] - df['cust_avg']) / df['cust_std'].replace(0, 1)).clip(-5, 5)
        df['amount_zscore_category'] = ((df['amount'] - df['cat_avg']) / df['cat_std'].replace(0, 1)).clip(-5, 5)
        df['customer_txn_count_log'] = np.log1p(df['cust_count'])
        
        # Encode categoricals (including region and vendor for audit)
        df['type_encoded'] = df['type'].map(TYPE_MAP).fillna(0)
        df['channel_encoded'] = df['channel'].map(CHANNEL_MAP).fillna(0)
        df['category_encoded'] = df['category'].map(CATEGORY_MAP).fillna(0)
        df['region_encoded'] = df['region'].map(REGION_MAP).fillna(0)
        df['vendor_encoded'] = df['vendor'].map(VENDOR_MAP).fillna(0)
        
        df = df.fillna(0)
        DATA_SOURCE = "clickhouse"
        print(f"Loaded {len(df):,} records from ClickHouse")
    else:
        raise Exception("Not enough data")
        
except Exception as e:
    print(f"Using synthetic data: {e}")
    # Generate synthetic data with region and vendor
    np.random.seed(42)
    n = 20000
    
    # Regions with different fraud patterns (APAC has lower thresholds)
    regions = np.random.choice(['APAC', 'EMEA', 'AMER'], n, p=[0.3, 0.35, 0.35])
    vendors = np.random.choice(['Visa', 'Mastercard', 'Amex'], n, p=[0.5, 0.35, 0.15])
    
    df = pd.DataFrame({
        'amount': np.abs(np.random.randn(n)) * 500,
        'tx_hour': np.random.randint(0, 24, n),
        'tx_day_of_week': np.random.randint(1, 8, n),
        'amount_zscore_customer': np.random.randn(n).clip(-5, 5),
        'amount_zscore_category': np.random.randn(n).clip(-5, 5),
        'customer_txn_count_log': np.random.uniform(0, 5, n),
        'type_encoded': np.random.randint(0, 4, n),
        'channel_encoded': np.random.randint(0, 4, n),
        'category_encoded': np.random.randint(0, 5, n),
        'region': regions,
        'vendor': vendors,
    })
    
    # Encode region and vendor
    df['region_encoded'] = df['region'].map(REGION_MAP).fillna(0)
    df['vendor_encoded'] = df['vendor'].map(VENDOR_MAP).fillna(0)
    
    # Create fraud labels based on patterns (region-aware like actual rules)
    # APAC: lower threshold ($8000), EMEA: medium ($10000), AMER: higher ($12000)
    df['is_fraud'] = (
        ((df['region'] == 'APAC') & (df['amount'] > 300)) |
        ((df['region'] == 'EMEA') & (df['amount'] > 400)) |
        ((df['region'] == 'AMER') & (df['amount'] > 500)) |
        (df['amount_zscore_customer'] > 2.5)
    ).astype(int)
    
    DATA_SOURCE = "synthetic"
    print(f"Generated {len(df):,} synthetic records")

print(f"\nFraud rate: {df['is_fraud'].mean()*100:.2f}%")
print(f"Features: {len(FEATURE_COLUMNS)} (including region_encoded, vendor_encoded for audit)")

In [ ]:
# Prepare training data
X = df[FEATURE_COLUMNS].values
y = df['is_fraud'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create DataFrames for signature
X_train_df = pd.DataFrame(X_train, columns=FEATURE_COLUMNS)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLUMNS)

print(f"Training set: {len(X_train):,} samples ({y_train.mean()*100:.2f}% fraud)")
print(f"Test set: {len(X_test):,} samples ({y_test.mean()*100:.2f}% fraud)")

In [ ]:
# Create MLflow Datasets for lineage tracking
# This enables reproducibility by tracking exact data versions used for training
import mlflow.data
from mlflow.data.pandas_dataset import PandasDataset
import os

# Create training dataset with labels included
train_data_with_labels = X_train_df.copy()
train_data_with_labels['is_fraud'] = y_train

# Create test dataset with labels included  
test_data_with_labels = X_test_df.copy()
test_data_with_labels['is_fraud'] = y_test

# Save datasets to parquet files for:
# 1. MLflow source tracking (file:// URIs work reliably)
# 2. Reference data for drift detection later
os.makedirs('/models/reference_data', exist_ok=True)
train_parquet_path = '/models/reference_data/train_dataset.parquet'
test_parquet_path = '/models/reference_data/test_dataset.parquet'

train_data_with_labels.to_parquet(train_parquet_path, index=False)
test_data_with_labels.to_parquet(test_parquet_path, index=False)
print(f"Saved reference datasets to /models/reference_data/")

# Create MLflow Dataset objects from parquet files
# These capture: schema, digest (hash), source path, and profile statistics
train_dataset: PandasDataset = mlflow.data.from_pandas(
    train_data_with_labels,
    source=train_parquet_path,
    name="fraud_training_data",
    targets="is_fraud"
)

test_dataset: PandasDataset = mlflow.data.from_pandas(
    test_data_with_labels,
    source=test_parquet_path,
    name="fraud_test_data",
    targets="is_fraud"
)

print(f"\nMLflow Datasets created for lineage tracking:")
print(f"  Training dataset: {train_dataset.name}")
print(f"    - Digest: {train_dataset.digest[:16]}...")
print(f"    - Source: {train_parquet_path}")
print(f"    - Samples: {len(train_data_with_labels):,}")
print(f"  Test dataset: {test_dataset.name}")
print(f"    - Digest: {test_dataset.digest[:16]}...")
print(f"    - Samples: {len(test_data_with_labels):,}")
print(f"\nData source: {DATA_SOURCE}")

## 2. Hyperparameter Experiments with MLflow Tracking

We'll run multiple experiments with different hyperparameters and track everything in MLflow.

In [ ]:
def train_and_log_model(
    params: Dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    run_name: str,
    train_dataset: PandasDataset = None,
    test_dataset: PandasDataset = None
) -> Tuple[str, float, xgb.XGBClassifier]:
    """
    Train a model with given parameters and log everything to MLflow.
    Uses JSON format for XGBoost models (best portability across versions).
    Returns run_id, f1_score, and trained model.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # Log experiment metadata
        mlflow.set_tags({
            "experiment_type": "hyperparameter_tuning",
            "data_source": DATA_SOURCE,
            "model_type": "xgboost",
            "use_case": "fraud_detection"
        })
        
        # Log datasets for lineage tracking (MLflow 2.4+)
        # This enables reproducibility by tracking exact data versions
        if train_dataset is not None:
            mlflow.log_input(train_dataset, context="training")
        if test_dataset is not None:
            mlflow.log_input(test_dataset, context="validation")
        
        # Log hyperparameters
        mlflow.log_params(params)
        mlflow.log_param("train_samples", len(X_train))
        mlflow.log_param("test_samples", len(X_test))
        mlflow.log_param("fraud_rate", f"{y_train.mean()*100:.2f}%")
        
        # Log feature lineage
        mlflow.log_dict({
            "features": FEATURE_COLUMNS,
            "descriptions": FEATURE_DESCRIPTIONS
        }, "feature_lineage.json")
        
        # Train model
        scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        
        model = xgb.XGBClassifier(
            **params,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric='logloss',
            early_stopping_rounds=10
        )
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )
        
        # Predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "auc_roc": roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0
        }
        
        # Log metrics
        mlflow.log_metrics(metrics)
        
        # Log feature importance
        importance = model.feature_importances_
        importance_dict = dict(zip(FEATURE_COLUMNS, importance.tolist()))
        mlflow.log_dict(importance_dict, "feature_importance.json")
        
        # Log feature importance as metrics for easy comparison
        for feat, imp in zip(FEATURE_COLUMNS, importance):
            mlflow.log_metric(f"importance_{feat}", imp)
        
        # Create feature importance plot
        fig, ax = plt.subplots(figsize=(10, 6))
        sorted_idx = np.argsort(importance)
        ax.barh(range(len(sorted_idx)), importance[sorted_idx])
        ax.set_yticks(range(len(sorted_idx)))
        ax.set_yticklabels([FEATURE_COLUMNS[i] for i in sorted_idx])
        ax.set_xlabel('Feature Importance')
        ax.set_title(f'Feature Importance - {run_name}')
        plt.tight_layout()
        
        # Save and log plot
        plot_path = f"/tmp/feature_importance_{run.info.run_id}.png"
        plt.savefig(plot_path, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(plot_path, "plots")
        plt.close()
        
        # Log confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        
        # Log model with signature and JSON format (best portability)
        signature = infer_signature(X_train_df, y_pred)
        mlflow.xgboost.log_model(
            model,
            artifact_path="model",
            signature=signature,
            input_example=X_train_df.iloc[:5],
            model_format="json"  # Best portability across XGBoost versions
        )
        
        print(f"  Run: {run_name}")
        print(f"  F1: {metrics['f1']:.4f}, AUC: {metrics['auc_roc']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}")
        
        return run.info.run_id, metrics['f1'], model

print("Training function defined with MLflow logging (JSON format for XGBoost)")
print("  - Now logs datasets with mlflow.log_input() for full lineage tracking")

In [ ]:
# Define hyperparameter grid for experiments
HYPERPARAMETER_EXPERIMENTS = [
    {
        "name": "baseline",
        "params": {
            "n_estimators": 100,
            "max_depth": 5,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    },
    {
        "name": "deeper_trees",
        "params": {
            "n_estimators": 100,
            "max_depth": 8,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    },
    {
        "name": "more_trees",
        "params": {
            "n_estimators": 200,
            "max_depth": 5,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    },
    {
        "name": "high_regularization",
        "params": {
            "n_estimators": 150,
            "max_depth": 4,
            "learning_rate": 0.1,
            "subsample": 0.7,
            "colsample_bytree": 0.7,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0
        }
    },
    {
        "name": "aggressive",
        "params": {
            "n_estimators": 300,
            "max_depth": 10,
            "learning_rate": 0.05,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        }
    }
]

print(f"Defined {len(HYPERPARAMETER_EXPERIMENTS)} experiments to run")

In [ ]:
# Run all experiments
print("="*70)
print("RUNNING HYPERPARAMETER EXPERIMENTS")
print("="*70)
print("  (with MLflow Dataset logging for full reproducibility)")
print("")

experiment_results = []

for exp in HYPERPARAMETER_EXPERIMENTS:
    run_name = f"exp_{exp['name']}_{datetime.now().strftime('%H%M%S')}"
    
    run_id, f1, model = train_and_log_model(
        params=exp['params'],
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        run_name=run_name,
        train_dataset=train_dataset,  # MLflow Dataset for lineage
        test_dataset=test_dataset      # MLflow Dataset for lineage
    )
    
    experiment_results.append({
        "name": exp['name'],
        "run_id": run_id,
        "f1_score": f1,
        "model": model,
        "params": exp['params']
    })

print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)

for r in sorted(experiment_results, key=lambda x: x['f1_score'], reverse=True):
    print(f"  {r['name']}: F1={r['f1_score']:.4f}")

print(f"\nDatasets logged with digests for reproducibility:")
print(f"  Training: {train_dataset.digest[:16]}...")
print(f"  Test: {test_dataset.digest[:16]}...")
print(f"\nView all experiments at: http://localhost:5001/#/experiments")

## 3. Feature Lineage and Importance Analysis

In [ ]:
# Get the best model
best_result = max(experiment_results, key=lambda x: x['f1_score'])
best_model = best_result['model']

print(f"Best Model: {best_result['name']}")
print(f"F1 Score: {best_result['f1_score']:.4f}")
print(f"Run ID: {best_result['run_id']}")

# Display feature importance with lineage
print("\n" + "="*70)
print("FEATURE IMPORTANCE & LINEAGE")
print("="*70)

importance = best_model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

for i, idx in enumerate(sorted_idx):
    feat = FEATURE_COLUMNS[idx]
    imp = importance[idx]
    desc = FEATURE_DESCRIPTIONS.get(feat, 'No description')
    print(f"\n{i+1}. {feat} (importance: {imp:.4f})")
    print(f"   {desc}")

In [ ]:
# Visualize feature importance comparison across experiments
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(FEATURE_COLUMNS))
width = 0.15

for i, result in enumerate(experiment_results):
    importance = result['model'].feature_importances_
    ax.bar(x + i*width, importance, width, label=result['name'])

ax.set_xlabel('Features')
ax.set_ylabel('Importance')
ax.set_title('Feature Importance Comparison Across Experiments')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(FEATURE_COLUMNS, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Register Best Model to MLflow Model Registry

Now we'll publish the best model to the MLflow Model Registry for production use.

In [ ]:
# Register the best model
print("="*70)
print("REGISTERING MODEL TO MLFLOW REGISTRY")
print("="*70)

model_uri = f"runs:/{best_result['run_id']}/model"

# Register model
try:
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )
    print(f"\nRegistered model: {MODEL_NAME}")
    print(f"Version: {model_version.version}")
    print(f"Run ID: {model_version.run_id}")
except Exception as e:
    print(f"Registration note: {e}")
    # Get latest version
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    if versions:
        model_version = versions[0]
        print(f"Using existing version: {model_version.version}")

In [ ]:
# Add model metadata and description
client.update_model_version(
    name=MODEL_NAME,
    version=model_version.version,
    description=f"""
Fraud Detection Model - {best_result['name']}

**Performance Metrics (Test Set):**
- F1 Score: {best_result['f1_score']:.4f}
- Training samples: {len(X_train):,}
- Test samples: {len(X_test):,}

**Hyperparameters:**
{json.dumps(best_result['params'], indent=2)}

**Features:** {len(FEATURE_COLUMNS)}
- {', '.join(FEATURE_COLUMNS[:6])}
- {', '.join(FEATURE_COLUMNS[6:])}

**Data Source:** {DATA_SOURCE}
**Trained:** {datetime.now().strftime('%Y-%m-%d %H:%M')}
"""
)

print(f"Model metadata updated")

In [ ]:
# Transition model to Production stage and set @champion alias
try:
    # Set Production stage
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=model_version.version,
        stage="Production",
        archive_existing_versions=True
    )
    print(f"Model transitioned to Production stage")
    
    # Set @champion alias (preferred way to reference production models)
    try:
        client.set_registered_model_alias(
            name=MODEL_NAME,
            alias="champion",
            version=model_version.version
        )
        print(f"Set @champion alias -> version {model_version.version}")
    except Exception as alias_err:
        print(f"Alias note: {alias_err}")
        
except Exception as e:
    print(f"Stage transition note: {e}")

# Store training metrics for drift detection
TRAINING_METRICS = {
    "f1": best_result['f1_score'],
    "accuracy": accuracy_score(y_test, best_model.predict(X_test)),
    "precision": precision_score(y_test, best_model.predict(X_test), zero_division=0),
    "recall": recall_score(y_test, best_model.predict(X_test), zero_division=0),
    "fraud_rate": float(y_train.mean()),
    "model_version": str(model_version.version),
    "trained_at": datetime.now().isoformat()
}

# Save training metrics for inference services
with open('/models/training_metrics.json', 'w') as f:
    json.dump(TRAINING_METRICS, f, indent=2)

print(f"\nTraining metrics saved for drift detection:")
for k, v in TRAINING_METRICS.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

print(f"\nModel can be loaded via:")
print(f"  - Stage: models:/{MODEL_NAME}/Production")
print(f"  - Alias: models:/{MODEL_NAME}@champion")
print(f"  - Version: models:/{MODEL_NAME}/{model_version.version}")
print(f"\nView model registry at: http://localhost:5001/#/models/{MODEL_NAME}")

## 5. Test Loading Model from Registry

Verify that inference services can load the model from MLflow.

In [ ]:
# Load model from registry (this is how inference services will load it)
print("Loading model from MLflow Registry...")
print("=" * 60)

# Three ways to load models:
# 1. By alias (recommended for production) - models:/name@alias
# 2. By stage - models:/name/Stage
# 3. By version - models:/name/version

# Method 1: Load via @champion alias (recommended)
model_uri_alias = f"models:/{MODEL_NAME}@champion"

# Method 2: Load via Production stage
model_uri_production = f"models:/{MODEL_NAME}/Production"

# Method 3: Load specific version
model_uri_version = f"models:/{MODEL_NAME}/{model_version.version}"

# Try loading via alias first (preferred)
for uri, method in [
    (model_uri_alias, "@champion alias"),
    (model_uri_production, "Production stage"),
    (model_uri_version, f"version {model_version.version}")
]:
    try:
        loaded_model = mlflow.xgboost.load_model(uri)
        print(f"Loaded model via {method}")
        print(f"  URI: {uri}")
        break
    except Exception as e:
        print(f"Could not load via {method}: {e}")
        continue

# Test prediction
test_prediction = loaded_model.predict(X_test[:5])
test_proba = loaded_model.predict_proba(X_test[:5])[:, 1]

print(f"\nTest predictions from loaded model:")
for i, (pred, prob) in enumerate(zip(test_prediction, test_proba)):
    print(f"  Sample {i+1}: {'FRAUD' if pred else 'SAFE'} ({prob:.2%} probability)")

print(f"\nInference services should use: models:/{MODEL_NAME}@champion")

## 6. Generate Inference Service Configuration

Generate configuration for inference services to use MLflow Registry.

In [ ]:
# Generate configuration for inference services
INFERENCE_CONFIG = {
    "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
    "model_name": MODEL_NAME,
    "model_stage": "Production",
    "model_version": model_version.version,
    "feature_columns": FEATURE_COLUMNS,
    "feature_descriptions": FEATURE_DESCRIPTIONS,
    "training_metrics": TRAINING_METRICS,
    "fraud_threshold": 0.5,
    "drift_alert_threshold": 0.1  # Alert if metrics deviate by 10%
}

# Save configuration
with open('/models/inference_config.json', 'w') as f:
    json.dump(INFERENCE_CONFIG, f, indent=2)

print("Inference configuration saved to /models/inference_config.json")
print("\nConfiguration:")
print(json.dumps(INFERENCE_CONFIG, indent=2, default=str))

## 7. Summary

### What We Accomplished:

1. **Dataset Lineage (NEW)**
   - Created MLflow Datasets with `mlflow.data.from_pandas()`
   - Logged datasets to runs with `mlflow.log_input()`
   - Each dataset has a unique digest (hash) for reproducibility
   - Saved reference data to `/models/reference_data/` for drift detection

2. **Experimentation Phase**
   - Ran 5 hyperparameter experiments
   - Tracked all parameters, metrics, and artifacts in MLflow
   - Logged feature importance for each experiment

3. **Feature Lineage**
   - Documented all 11 features with descriptions
   - Tracked feature importance across experiments
   - Logged feature lineage JSON for each run

4. **Model Publishing**
   - Registered best model to MLflow Model Registry
   - Added model metadata and description
   - Transitioned to Production stage

5. **Production Configuration**
   - Saved inference configuration with training metrics
   - Configured drift detection thresholds
   - Verified model loading from registry

### Dataset Tracking Benefits:

| Feature | Benefit |
|---------|---------|
| **Digest** | Unique hash identifies exact data version |
| **Parquet Files** | Reference data for drift detection |
| **Schema** | Column names and types auto-captured |
| **Context** | "training" vs "validation" labels |
| **Reproducibility** | Re-run experiments with identical data |

### Reference Data for Drift Detection:
```
/models/reference_data/
  - train_dataset.parquet  (training features + labels)
  - test_dataset.parquet   (test features + labels)
```

### Next Steps:

The inference services (Kafka, PGAA, ClickHouse, RisingWave) will:
1. Load model from MLflow Registry (not pkl files)
2. Track inference metrics and compare to training metrics
3. Detect and alert on model drift
4. (Future) Compare production data against reference parquet files

### URLs:
- **MLflow UI**: http://localhost:5001
- **Experiments**: http://localhost:5001/#/experiments
- **Model Registry**: http://localhost:5001/#/models
- **Datasets**: View in run details under "Datasets" tab